# Librerías y configuración inicial

In [1]:
import time
import json
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from OMIEData.Enums.all_enums import SystemType
from OMIEData.DataImport.omie_marginalprice_importer import OMIEMarginalPriceFileImporter
from OMIEData.DataImport.omie_energy_by_technology_importer import OMIEEnergyByTechnologyImporter

BASE_URL = "https://apidatos.ree.es/es/datos"

START_DATE = datetime(2019, 1, 1, 0, 0)
END_DATE   = datetime(2025, 12, 31, 23, 59)

TIMEOUT     = 20
MAX_RETRIES = 3
MAX_WORKERS = 6

session = requests.Session()

PARAMS_GEO = {
    "geo_trunc" : "electric_system",
    "geo_limit" : "peninsular",
    "geo_ids"   : "8741"
}

# Obtención de datos desde Red Eléctrica Española (REE)

In [2]:
def request_with_fallback(url, params, start, end, depth=0):
    try:
        r = session.get(url, params=params, timeout=TIMEOUT)
    except requests.RequestException as e:
        print(f"[ERROR] Conexión: {e}")
        return None

    if r.status_code == 200:
        return r.json()

    msg = r.text
    print(f"[WARNING] {r.status_code}: {msg}")

    if r.status_code in [429, 500, 502, 503]:
        for attempt in range(1, MAX_RETRIES + 1):
            time.sleep(2 ** attempt)
            try:
                r = session.get(url, params=params, timeout=TIMEOUT)
                if r.status_code == 200:
                    return r.json()
            except:
                pass

    # -----------------------------
    # Error típico REE, manda error 400 con el siguiente mensaje:
    #   "Los datos solicitados no están disponibles en este momento. Inténtelo de nuevo más tarde."
    # Intentamos de nuevo dividiendo el horizonte temporal en dos, por si es demasiado grande para la API
    # -----------------------------
    if "Los datos solicitados" in msg.lower() and depth < 3:
        mid   = start + (end - start) / 2
        left  = request_with_fallback(url, params, start, mid, depth + 1)
        right = request_with_fallback(url, params, mid, end, depth + 1)

        results = []
        if left: results.append(left)
        if right: results.append(right)

        return results if results else None
    
    return None

In [3]:
def generate_ranges(start, end, time_trunc):
    ranges  = []
    current = start

    while current <= end:
        if time_trunc == "hour":
            next_date = current + pd.DateOffset(months=1)
        else:
            next_date = current + pd.DateOffset(years=1)
        
        next_date = min(next_date - timedelta(days=1), end)
        ranges.append((current, next_date))
        current = next_date + timedelta(days=1)

    return ranges

In [4]:
def fetch_endpoint(endpoint, time_trunc, use_geo=True):
    url = f"{BASE_URL}/{endpoint}"

    ranges = generate_ranges(START_DATE, END_DATE, time_trunc)

    print(f"Endpoint: {endpoint}")
    print(f"Total chunks: {len(ranges)}")

    results = []

    def task(s, e):
        params = {
            "start_date" : s.strftime("%Y-%m-%dT00:00"),
            "end_date"   : e.strftime("%Y-%m-%dT23:59"),
            "time_trunc" : time_trunc
        }

        if use_geo:
            params.update(PARAMS_GEO)

        return request_with_fallback(url, params, s, e)

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(task, s, e) for s, e in ranges]

        for future in as_completed(futures):
            res = future.result()

            if res is None:
                continue

            if isinstance(res, list):
                results.extend(res)
            else:
                results.append(res)

    print(f"Chunks obtenidos con éxito: {len(results)}\n")

    return results

In [5]:
print(f"Rango de fechas de los datos: {START_DATE.strftime('%Y-%m-%d')} - {END_DATE.strftime('%Y-%m-%d')}\n")
json_demanda_evolucion = fetch_endpoint("demanda/evolucion",                    time_trunc="hour",  use_geo=True)
json_frontera_fisicos  = fetch_endpoint("intercambios/todas-fronteras-fisicos", time_trunc="day",   use_geo=False)
json_balance           = fetch_endpoint("balance/balance-electrico",            time_trunc="day",   use_geo=True)

Rango de fechas de los datos: 2019-01-01 - 2025-12-31

Endpoint: demanda/evolucion
Total chunks: 84
Chunks obtenidos con éxito: 84

Endpoint: intercambios/todas-fronteras-fisicos
Total chunks: 7
Chunks obtenidos con éxito: 7

Endpoint: balance/balance-electrico
Total chunks: 7
Chunks obtenidos con éxito: 7



In [6]:
def save_file(data, filename, extension='csv'):

    current = Path.cwd()
    path = current.parent / "data" / "raw_data" / extension

    # Creamos la carpeta si no existe
    path.mkdir(parents=True, exist_ok=True)

    file_path = path / f"{filename}.json" if extension == 'json' else path / f"{filename}.csv"

    if extension == 'json':
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False)
    else:
        data.to_csv(file_path, index=False, sep=";")

    print(f"[INFO] Archivo guardado con éxito en la siguiente ruta: {file_path}")

In [7]:
save_file(json_demanda_evolucion, 'demanda',      extension='json')
save_file(json_balance,           'balance',      extension='json')
save_file(json_frontera_fisicos,  'intercambios', extension='json')

[INFO] Archivo guardado con éxito en la siguiente ruta: c:\UOC\TFM\data\raw_data\json\demanda.json
[INFO] Archivo guardado con éxito en la siguiente ruta: c:\UOC\TFM\data\raw_data\json\balance.json
[INFO] Archivo guardado con éxito en la siguiente ruta: c:\UOC\TFM\data\raw_data\json\intercambios.json


# Obtención de datos desde Operador del Mercado Ibérico de Energía (OMIE)

In [8]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [9]:
def generate_daily_ranges(start, end):
    current = start
    while current <= end:
        yield current, current
        current += timedelta(days=1)

def fetch_omie_day(start, end):
    df_price = None
    df_tech  = None

    # Precio marginal
    try:
        df_price = OMIEMarginalPriceFileImporter(date_ini=start, date_end=end).read_to_dataframe(verbose=False)
    except Exception as e:
        print(f"[ERROR PRICE] {start.date()} : {e}")
    
    # Energía (casada) por tecnología
    try:
        df_tech  = OMIEEnergyByTechnologyImporter(date_ini=start, date_end=end, system_type=SystemType.SPAIN).read_to_dataframe(verbose=False)
        return df_price, df_tech
    except Exception as e:
        print(f"[ERROR TECH] {start.date()} : {e}")
    
    return df_price, df_tech


def fetch_omie_parallel(start, end):
    ranges = list(generate_daily_ranges(start, end))

    prices = []
    techs  = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    
        futures = {executor.submit(fetch_omie_day, s, e): (s, e) for s, e in ranges}

        for i, future in enumerate(as_completed(futures), 1):
            df_price, df_tech = future.result()

            if df_price is not None and not df_price.empty:
                prices.append(df_price)

            if df_tech is not None and not df_tech.empty:
                techs.append(df_tech)

            if i % 50 == 0 or i == len(ranges):
                print(f"[{i}/{len(ranges)}] días completados")

    df_price_final = pd.concat(prices, ignore_index=True) if prices else pd.DataFrame()
    df_tech_final  = pd.concat(techs, ignore_index=True) if techs else pd.DataFrame()

    return df_price_final, df_tech_final

In [10]:
df_price, df_tech = fetch_omie_parallel(start=START_DATE, end=END_DATE)

[50/2557] días completados
[100/2557] días completados
[150/2557] días completados
[200/2557] días completados
[250/2557] días completados
[300/2557] días completados
[350/2557] días completados
[400/2557] días completados
[450/2557] días completados
[500/2557] días completados
[550/2557] días completados
[600/2557] días completados
[650/2557] días completados
[700/2557] días completados
[750/2557] días completados
[800/2557] días completados
[850/2557] días completados
[900/2557] días completados
[950/2557] días completados
[1000/2557] días completados
[1050/2557] días completados
[1100/2557] días completados
[1150/2557] días completados
[1200/2557] días completados
[1250/2557] días completados
[1300/2557] días completados
[1350/2557] días completados
[1400/2557] días completados
[1450/2557] días completados
[1500/2557] días completados
[1550/2557] días completados
[1600/2557] días completados
[1650/2557] días completados
[1700/2557] días completados
[1750/2557] días completados
[1800

In [11]:
save_file(df_price, 'precio_marginal', extension='csv')
save_file(df_tech,  'energia_casada',  extension='csv')

[INFO] Archivo guardado con éxito en la siguiente ruta: c:\UOC\TFM\data\raw_data\csv\precio_marginal.csv
[INFO] Archivo guardado con éxito en la siguiente ruta: c:\UOC\TFM\data\raw_data\csv\energia_casada.csv
